# Tydzień 6: Klasyfikacja – porównanie modeli, macierz pomyłek i granice decyzyjne 🧭

| | |
|---|---|
| **Rozdział książki** | 3. *Classification* – podrozdziały: *Comparing Classification Models*, *Creating the Model*, *Tuning the Model*, *Making Predictions*, *Plotting the Model*, *Finalizing and Saving the Model* |
| **Czas zajęć** | 90 minut |
| **Dane** | `iris` – pomiary 150 kwiatów irysa (wbudowane w PyCaret) |
| **Nowe pojęcia** | dokładność (*accuracy*), macierz pomyłek (*confusion matrix*), granica decyzyjna (*decision boundary*) |

## 🎯 Cel zajęć
Po dzisiejszych zajęciach będziesz:
1. umieć porównać klasyfikatory funkcją `compare_models()` i odczytać metryki klasyfikacji,
2. umieć zbudować, dostroić i sprawdzić model na zbiorze testowym,
3. czytać **macierz pomyłek** – „kto z kim został pomylony”,
4. rozumieć, czym są **granice decyzyjne** i dlaczego różne modele rysują je inaczej,
5. zapisać klasyfikator i rozpoznać gatunek **nowego** kwiatu.

## 🗓️ Plan zajęć
| Czas | Co robimy |
|---|---|
| 0–15 min | 📚 Teoria: metryki, macierz pomyłek, granice decyzyjne |
| 15–65 min | 👣 Krok po kroku |
| 65–85 min | 🧑‍💻 Ćwiczenie: model KNN – ✅ **obowiązkowe** (zaliczenie) |
| 85–90 min | 📝 Podsumowanie |
| poza zajęciami | 🏋️ Część 4: ćwiczenia dodatkowe (trudniejsze, nieobowiązkowe) – dla szybszych osób lub do domu |

---
# 📚 Część 1: Wprowadzenie teoretyczne

## Dokładność (*Accuracy*) ✅
Najprostsza metryka klasyfikacji: **jaka część odpowiedzi była poprawna?**

$$\text{Accuracy} = \frac{\text{liczba poprawnych odpowiedzi}}{\text{liczba wszystkich odpowiedzi}}$$

Jeśli model dobrze rozpoznał 29 z 30 kwiatów, dokładność wynosi 29/30 ≈ **96,7%**.
⚠️ Dokładność ma sens głównie przy **zbalansowanych klasach** (jak w irysach).

## Inne metryki w tabeli PyCaret (na razie tylko dla ciekawych)
| Metryka | Pytanie, na które odpowiada |
|---|---|
| **AUC** | jak dobrze model „układa w kolejności” przykłady wg pewności (1 = idealnie) |
| **Recall** (czułość) | ile przykładów danej klasy model **znalazł**? (np. ilu chorych wykrył) |
| **Precision** (precyzja) | ile odpowiedzi „to klasa X” było **trafnych**? |
| **F1** | kompromis (średnia) między precyzją a czułością |
| **Kappa**, **MCC** | dokładność „poprawiona” o trafienia przypadkowe |

## Macierz pomyłek (*Confusion Matrix*) 🤔
Tabela, która pokazuje, **kto z kim został pomylony**. Wiersze = **prawdziwa** klasa, kolumny = klasa **przewidziana** przez model.
Liczby na przekątnej to trafienia, a wszystko poza przekątną – pomyłki.

|  | przewidziano: setosa | przewidziano: versicolor | przewidziano: virginica |
|---|---|---|---|
| **naprawdę: setosa** | ✅ 10 | 0 | 0 |
| **naprawdę: versicolor** | 0 | ✅ 9 | ❌ 1 |
| **naprawdę: virginica** | 0 | 0 | ✅ 10 |

Tu model raz wziął *versicolor* za *virginicę* – i od razu wiemy, **które** gatunki sprawiają mu kłopot.

## Granica decyzyjna (*Decision Boundary*) 🗺️
Wyobraź sobie mapę, na której model rysuje **granice państw**: każdy punkt na mapie należy do jakiegoś „kraju” (klasy).
Nowy kwiat dostaje gatunek „kraju”, w którym wylądował. Różne modele rysują różne granice:
- **LDA** (liniowa analiza dyskryminacyjna) i regresja logistyczna – granice to **proste linie**,
- **drzewo decyzyjne** – granice „schodkowe”, złożone z poziomych i pionowych odcinków,
- **KNN** (k-najbliższych sąsiadów) – granice „poszarpane”, bo kwiat dostaje gatunek większości swoich najbliższych sąsiadów.

## Linear Discriminant Analysis (LDA) w jednym zdaniu
LDA szuka takiego „kierunku patrzenia” na dane, z którego klasy są **jak najlepiej rozsunięte**, a potem dzieli przestrzeń prostymi liniami.
(Uwaga: skrót LDA w tygodniu 9 będzie oznaczał zupełnie inny model – *Latent Dirichlet Allocation*!)

---
# 👣 Część 2: Krok po kroku

### Krok 1: Powtórka – dane i `setup()`

In [ ]:
# Wyłączamy ostrzeżenia (warnings), żeby nie zaśmiecały ekranu.
# Ostrzeżenie to NIE błąd – to tylko podpowiedź dla programistów bibliotek.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Importy: tabele, dane i wszystkie funkcje KLASYFIKACJI PyCaret
import pandas as pd
from pycaret.datasets import get_data
from pycaret.classification import *

In [ ]:
# Wczytujemy irysy (bez wyświetlania tabeli)
data = get_data('iris', verbose=False)

In [ ]:
# Ten sam setup co tydzień temu; verbose=False ukrywa tabelę ustawień
classf = setup(data=data, target='species', train_size=0.8, normalize=True, session_id=3934, verbose=False)

### Krok 2: Porównanie klasyfikatorów – `compare_models()` 🏁
⏳ Potrwa ok. 1 minuty. Każdy model jest oceniany 10-krotną walidacją krzyżową.

In [ ]:
# Ranking według dokładności (Accuracy); najlepszy model trafia do zmiennej best
best = compare_models(sort='Accuracy')

In [ ]:
# Najlepszy model
print(best)

🔎 Na samej górze znajdują się modele z dokładnością ok. **97–98%**, m.in. **Linear Discriminant Analysis** (`lda`), regresja logistyczna (`lr`)
i KNN (`knn`). Na dole jest `dummy` – model „na ślepo” z dokładnością ok. 33% (zgadywanie jednej z trzech klas).
Za książką wybierzemy **LDA**: jest prosty, szybki i bardzo dokładny.

⚠️ **Dlaczego przy niektórych modelach AUC = 0.0000?** To **znany błąd** tej wersji PyCaret (3.3) przy klasyfikacji
wieloklasowej – dla części modeli nie udaje się policzyć AUC i wpisywane jest 0. Model **nie jest** zepsuty!
Dlatego patrzymy na **dokładność** (*Accuracy*), która jest liczona poprawnie.

### Krok 3: Budujemy model LDA – `create_model()`

In [ ]:
# 'lda' = Linear Discriminant Analysis; walidacja krzyżowa warstwowa (StratifiedKFold), 10 części
model = create_model('lda')

🔎 W większości części (*folds*) dokładność wynosi **1.0000**, czyli 100% – model nie pomylił się ani razu.
W kilku częściach jedna pomyłka na 12 kwiatów obniża wynik do ok. 0,92.

### Krok 4: Egzamin na zbiorze testowym – `predict_model()` 📝

In [ ]:
# Predykcje dla 30 kwiatów ze zbioru testowego (model ich wcześniej nie widział)
predictions = predict_model(model)

In [ ]:
# Kolumny: species (prawda), prediction_label (przewidywany gatunek), prediction_score (pewność modelu 0–1)
predictions.head(10)

In [ ]:
# Ile przewidywań było trafnych? (True = trafione, False = pomyłka)
(predictions['species'] == predictions['prediction_label']).value_counts()

### Krok 5: Macierz pomyłek 🤔
Uwaga: na wykresie klasy są oznaczone **liczbami** – pamiętasz kodowanie etykiet z tygodnia 5?
`0 = Iris-setosa`, `1 = Iris-versicolor`, `2 = Iris-virginica`.

In [ ]:
# Wiersze (True Class) = prawdziwy gatunek, kolumny (Predicted Class) = gatunek przewidziany przez model
plot_model(model, plot='confusion_matrix')

🔎 Prawie wszystkie liczby leżą na **przekątnej** (trafienia). Jeśli pojawia się pomyłka, to między klasami **1 i 2**
(*versicolor* / *virginica*) – tymi, które nachodziły na siebie na wykresach w tygodniu 5. *Setosa* (0) jest rozpoznawana bezbłędnie.

In [ ]:
# Raport klasyfikacji: precyzja, czułość i F1 dla każdej klasy osobno (im bardziej zielono/ciemno, tym lepiej)
plot_model(model, plot='class_report')

### Krok 6: Granice decyzyjne 🗺️
Kwiat ma 4 cechy, a ekran tylko 2 wymiary. Dlatego PyCaret najpierw „spłaszcza” dane do 2 wymiarów
(techniką PCA – poznamy ją za tydzień) i dopiero wtedy rysuje mapę z granicami.

In [ ]:
# Granice decyzyjne modelu LDA: kolorowe obszary = "kraje" klas, punkty = kwiaty
plot_model(model, plot='boundary')

🔎 Granice LDA to **proste linie**. Jeden gatunek (0 – *setosa*) leży daleko od reszty, a dwa pozostałe stykają się ze sobą.
Dla porównania zbudujmy **drzewo decyzyjne** (*Decision Tree*):

In [ ]:
# 'dt' = Decision Tree Classifier – drzewo decyzyjne (seria pytań "tak/nie" o cechy)
dt = create_model('dt')

In [ ]:
# Granice drzewa są "schodkowe" – złożone z poziomych i pionowych odcinków
plot_model(dt, plot='boundary')

### Krok 7: Strojenie modelu – `tune_model()` 🎛️
Drzewo decyzyjne ma kilka hiperparametrów (np. maksymalną głębokość). Dostroimy je **przeszukiwaniem losowym**.
💡 W książce autor używa biblioteki `scikit-optimize` (optymalizacja bayesowska) i modelu CatBoost – to działa podobnie,
ale trwa dużo dłużej, więc na zajęciach używamy szybszej wersji.

In [ ]:
# Sprawdzamy 20 losowych kombinacji ustawień (domyślna siatka PyCaret), optymalizujemy dokładność
tuned_dt = tune_model(dt, optimize='Accuracy', n_iter=20)

In [ ]:
# Najlepsze znalezione ustawienia drzewa
print(tuned_dt)

🔎 Jeśli zobaczysz komunikat *„Original model was better than the tuned model, hence it will be returned”* – strojenie
nic nie poprawiło i PyCaret **zwrócił oryginalne drzewo** (pamiętasz `choose_better` z tygodnia 4?).
Przy tak małym i łatwym zbiorze to zupełnie normalne: domyślne ustawienia są już bardzo dobre.

### Krok 8: Finalizacja, zapis i rozpoznawanie nowego kwiatu 💾🌸
Zostajemy przy zwycięzcy – modelu **LDA**.

In [ ]:
# Trenujemy LDA na wszystkich 150 kwiatach
final_model = finalize_model(model)

In [ ]:
# Zapisujemy model do ../modele/classification_model.pkl (średnik ukrywa długi wydruk)
import os
os.makedirs('../modele', exist_ok=True)
save_model(final_model, '../modele/classification_model');

In [ ]:
# Wczytujemy model z pliku
loaded_model = load_model('../modele/classification_model')

In [ ]:
# Nowy kwiat znaleziony w ogrodzie – mierzymy go linijką (w cm)
new_flower = pd.DataFrame([{'sepal_length': 6.3, 'sepal_width': 2.9, 'petal_length': 5.6, 'petal_width': 1.8}])
predict_model(loaded_model, data=new_flower)

🔎 `prediction_label` to rozpoznany gatunek, a `prediction_score` – **pewność** modelu (1.0 = 100%).

---
# 🧑‍💻 Część 3: Ćwiczenie dla Ciebie
> ✅ **Ćwiczenie obowiązkowe** – na jego podstawie zaliczasz dzisiejsze zajęcia (zaliczone / niezaliczone).

**Zadanie: Model k-najbliższych sąsiadów (KNN)** 👥

KNN działa jak zasada „powiedz mi, kim są twoi sąsiedzi, a powiem ci, kim jesteś”: nowy kwiat dostaje gatunek, który
dominuje wśród jego *k* najbliższych sąsiadów (domyślnie k = 5).

- **a)** Zbuduj model KNN (ID: `'knn'`) i zapisz go w zmiennej `knn`.
- **b)** Sprawdź go na zbiorze testowym (`predict_model`). Jaka jest dokładność (*Accuracy*)?
- **c)** Narysuj macierz pomyłek i granice decyzyjne. Czym granice KNN różnią się od granic LDA?
- **d)** Rozpoznaj modelem `loaded_model` (LDA) gatunek kwiatu: działka 5.0 × 3.4 cm, płatek 1.5 × 0.2 cm.

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- a) `create_model('knn')`
- b) `predict_model(knn)` – dokładność jest w tabelce nad wynikami.
- c) `plot_model(knn, plot='confusion_matrix')` i `plot_model(knn, plot='boundary')`
- d) Utwórz `pd.DataFrame([{...}])` z kolumnami `sepal_length`, `sepal_width`, `petal_length`, `petal_width`.

</details>

In [ ]:
# ✍️ Twój kod tutaj (możesz dodawać kolejne komórki przyciskiem "+ Code")

### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 🏋️ Część 4: Ćwiczenia dodatkowe – trochę trudniejsze
Dla osób, które skończyły ćwiczenie podstawowe, albo **do samodzielnej pracy w domu**.
Ćwiczenia z tej części są **nieobowiązkowe** – nie są wymagane do zaliczenia zajęć.

Te zadania mają mniej podpowiedzi w treści, łączą kilka poznanych narzędzi i czasem wprowadzają **jedną nową funkcję**
(zawsze opisaną w podpowiedzi). Poziomy trudności: ⭐⭐ – średnie, ⭐⭐⭐ – wyzwanie.

## ⭐⭐ Ćwiczenie 2: Najtrudniejsza para – versicolor kontra virginica
Macierz pomyłek pokazała, że model myli tylko *versicolor* z *virginicą*. Zbudujmy klasyfikator **binarny** tylko dla tej pary.

- **a)** Utwórz tabelę `vv` bez kwiatów *Iris-setosa* i uruchom dla niej `setup()` (80/20, normalizacja, `session_id=3934`).
  Jaki typ zadania (`Target type`) rozpoznał PyCaret?
- **b)** Porównaj modele (`compare_models`, sortowanie po dokładności).
- **c)** Zbuduj regresję logistyczną (`'lr'`), narysuj jej **macierz pomyłek** oraz **krzywą ROC** na zbiorze testowym.

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Filtrowanie: `data.query("species != 'Iris-setosa'")`.
- **Krzywa ROC** (nowy wykres): `plot_model(model, plot='auc')`. Pokazuje, jak zmienia się odsetek wykrytych przypadków
  (oś Y) kosztem fałszywych alarmów (oś X) przy różnych progach decyzji. **Pole pod krzywą (AUC)**: 1 = ideał, 0,5 = zgadywanie.
- W klasyfikacji **binarnej** AUC liczy się poprawnie (błąd z AUC = 0 dotyczy tylko wielu klas).

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

## ⭐⭐⭐ Ćwiczenie 3: Cukrzyca – gdy dokładność to za mało
Przejdźmy do poważniejszych danych: zbiór `diabetes` (768 pacjentek, cel: `Class variable`, 1 = cukrzyca).

- **a)** Przygotuj dane (`session_id=123`) i porównaj modele `'lr'`, `'lda'`, `'knn'`, `'dt'`, `'rf'`, `'nb'` według dokładności.
- **b)** Zbuduj najlepszy model (regresję logistyczną `'lr'`) i sprawdź go na zbiorze testowym. **Ilu chorych model przeoczył?**
- **c)** Domyślnie model mówi „chora”, gdy prawdopodobieństwo > 0,5. **Obniż próg do 0,3** i porównaj czułość (*Recall*),
  precyzję i liczbę przeoczonych chorych.
- **d)** Który próg wybrałbyś(-abyś) w badaniu przesiewowym? Dlaczego?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Dane: `get_data('diabetes', verbose=False)`; nowy `setup()` zastąpi poprzedni eksperyment.
- **Przeoczeni chorzy** (*false negatives*): prawdziwa klasa 1, przewidziana 0. Policzysz ich tabelą krzyżową:
  `pd.crosstab(pred['Class variable'], pred['prediction_label'])`.
- Nowy parametr: `predict_model(model, probability_threshold=0.3)` zmienia próg decyzji (działa w klasyfikacji binarnej).

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 📝 Podsumowanie
- `compare_models(sort='Accuracy')` porównuje klasyfikatory; w irysach wygrywają modele z dokładnością ok. 97–98%.
- `predict_model()` zwraca przewidywany gatunek (`prediction_label`) i pewność modelu (`prediction_score`).
- **Macierz pomyłek** pokazuje, które klasy model myli (u nas: *versicolor* ↔ *virginica*).
- **Granice decyzyjne** dzielą przestrzeń na „kraje” klas: proste (LDA), schodkowe (drzewo), poszarpane (KNN).
- `finalize_model()` + `save_model()` zapisują gotowy klasyfikator, `load_model()` wczytuje go do użycia.

## 📖 Słowniczek PL / EN
| Polski | English | Znaczenie |
|---|---|---|
| dokładność | accuracy | odsetek poprawnych odpowiedzi |
| macierz pomyłek | confusion matrix | tabela: prawdziwa klasa × przewidziana klasa |
| precyzja | precision | ile wskazań „klasa X” było trafnych |
| czułość | recall / sensitivity | ile przykładów klasy X model znalazł |
| miara F1 | F1 score | kompromis między precyzją a czułością |
| granica decyzyjna | decision boundary | linia oddzielająca obszary różnych klas |
| liniowa analiza dyskryminacyjna | Linear Discriminant Analysis (LDA) | model z prostymi granicami między klasami |
| drzewo decyzyjne | decision tree | model zadający serię pytań „tak/nie” |
| k-najbliższych sąsiadów | k-Nearest Neighbors (KNN) | klasa = większość wśród k sąsiadów |
| pewność predykcji | prediction score / probability | jak bardzo model jest pewny odpowiedzi |

➡️ **Za tydzień:** **uczenie nienadzorowane** – algorytm K-Means sam znajdzie grupy w danych, bez żadnych etykiet!